# Raw Reaction Synonym Enrichment Walkthrough

This notebook demonstrates the unmapped-name synonym enrichment workflow: collect names from raw reactions, ask for synonym candidates, vote by normalized synonym, review an audit file, and apply approved additions.

By default it uses deterministic demo responses and does **not** call an LLM or mutate compound data. Flip the safety switches in the setup cell only when you intentionally want to run the real mining/apply workflow.

## 1. Setup

The notebook imports the real service used by the CLI. It can be launched from the repo root or from the `notebooks/` directory.

In [1]:
from pathlib import Path
from pprint import pprint
import json
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.chemy import Chemy
from scripts.ops.raw_reactions.synonym_enrichment import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MIN_VOTES,
    DEFAULT_ROUNDS,
    RawReactionSynonymEnricher,
    build_synonym_prompt,
    parse_synonym_jsonl,
)

DATA_DIR = REPO_ROOT / "data"
chemy = Chemy(str(DATA_DIR))
enricher = chemy.raw_reaction_synonyms

# Safety switches. Keep these False for inspection/demo runs.
RUN_REAL_LLM = True
RUN_APPLY = False
PRESET_NAME = "wiki_crc_rp"  # Set to None to scan every raw reaction preset.
REAL_MODEL = None            # None uses the service default model.

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")
print(f"Defaults: rounds={DEFAULT_ROUNDS}, min_votes={DEFAULT_MIN_VOTES}, batch_size={DEFAULT_BATCH_SIZE}")

Repo root: /home/canary/Documents/Code/chemy
Data dir:  /home/canary/Documents/Code/chemy/data
Defaults: rounds=5, min_votes=3, batch_size=7


## 2. Collect Unmapped Names

The collector reads raw reaction JSONL files and uses `ReactionLLMParser.parse_structured_reaction`, so “unmapped” means exactly what currently prevents raw reactions from parsing into CID-backed reactions.

In [2]:
unmapped = enricher.collect_unmapped_names(PRESET_NAME, limit=20)
print(f"Top unmapped participant names: {len(unmapped)} shown")
for entry in unmapped[:10]:
    variants = ", ".join(
        f"{variant['name']} ({variant['count']})"
        for variant in entry["raw_variants"][:3]
    )
    print(f"{entry['count']:>4}  {entry['name']}  [{variants}]")

Top unmapped participant names: 20 shown
 319  hydrogen sulfide  [hydrogen sulfide (233), Hydrogen sulfide (86)]
 149  pyridinium chlorochromate  [pyridinium chlorochromate (98), Pyridinium chlorochromate (51)]
 109  hydrazine hydrate  [hydrazine hydrate (70), Hydrazine hydrate (39)]
  85  hydroxylamine hydrochloride  [hydroxylamine hydrochloride (56), Hydroxylamine hydrochloride (29)]
  33  borane-tetrahydrofuran complex  [borane-tetrahydrofuran complex (24), Borane-tetrahydrofuran complex (9)]
  30  copper(II) sulfide  [copper(II) sulfide (20), Copper(II) sulfide (10)]
  30  D-glucose  [D-glucose (22), D-Glucose (8)]
  30  fluorine gas  [fluorine gas (23), Fluorine gas (7)]
  29  cobalt(II) hydroxide  [cobalt(II) hydroxide (21), Cobalt(II) hydroxide (8)]
  26  ammonia solution  [ammonia solution (19), Ammonia solution (7)]


## 3. Inspect The Prompt

The production prompt is intentionally compact. The LLM receives a JSONL list of names and must return one JSON object per input line with `name` and `synonyms`.

In [3]:
demo_names = [entry["name"] for entry in unmapped[:5]] or ["Grain spirit", "Wood alcohol"]
print(build_synonym_prompt(demo_names))

For each input chemical name, list common equivalent names for the same compound only. Use names that could identify the compound in a reaction.
Return exactly one JSON object per input line, in the same order:
{"name":"<input name>","synonyms":["<name1>","<name2>"]}
Rules:
- If unsure, use an empty synonyms list.
- Return only JSONL, no extra text.

{"name": "hydrogen sulfide"}
{"name": "pyridinium chlorochromate"}
{"name": "hydrazine hydrate"}
{"name": "hydroxylamine hydrochloride"}
{"name": "borane-tetrahydrofuran complex"}


## 4. Parse And Vote By Normalized Synonym

Votes are counted by normalized synonym, not raw spelling. Duplicate spellings in one response count once for that round, while raw spellings are still retained for audit readability.

In [4]:
demo_entry = {
    "name": "Grain spirit",
    "norm_name": "grainspirit",
    "count": 3,
    "raw_variants": [{"name": "Grain spirit", "count": 3}],
    "examples": [],
}

demo_responses = [
    json.dumps({"name": "Grain spirit", "synonyms": ["Ethyl alcohol", "ethyl alcohol"]}),
    json.dumps({"name": "Grain spirit", "synonyms": ["ethyl  alcohol"]}),
    json.dumps({"name": "Grain spirit", "synonyms": ["Methanol"]}),
]

for i, response in enumerate(demo_responses, start=1):
    parsed = parse_synonym_jsonl(
        response,
        {demo_entry["norm_name"]},
        chemy.compounds.normalize_chem_name,
    )
    print(f"Round {i}:")
    pprint(dict(parsed))

Round 1:
{'grainspirit': ['Ethyl alcohol', 'ethyl alcohol']}
Round 2:
{'grainspirit': ['ethyl  alcohol']}
Round 3:
{'grainspirit': ['Methanol']}


In [5]:
class DemoLLMClient:
    def __init__(self, responses):
        self.responses = list(responses)
        self.completion_tokens_total = 0

    def fetch_answer_str(self, *args, **kwargs):
        return self.responses.pop(0)

    def get_future_result(self, future, executor):
        return future.result()


demo_enricher = RawReactionSynonymEnricher(
    str(DATA_DIR),
    chemy.compounds,
    chemy.store,
    chemy.logger,
    DemoLLMClient(demo_responses),
    chemy.reaction_llm,
)

demo_audit = demo_enricher._mine_batch([demo_entry], model="demo-model", rounds=3, min_votes=2)
pprint(demo_audit[0])

{'add_synonym': None,
 'approved': False,
 'candidate_synonyms': [{'decision': 'matched',
                         'matched_cid': 702,
                         'matched_name': 'Ethanol',
                         'norm_synonym': 'ethylalcohol',
                         'raw_synonyms': [{'count': 2,
                                           'synonym': 'ethyl alcohol'},
                                          {'count': 1,
                                           'synonym': 'Ethyl alcohol'}],
                         'synonym': 'ethyl alcohol',
                         'votes': 2}],
 'count': 3,
 'examples': [],
 'name': 'Grain spirit',
 'norm_name': 'grainspirit',
 'raw_variants': [{'count': 3, 'name': 'Grain spirit'}],
 'status': 'max_synonyms',
 'target_cid': None,
 'target_name': None}


## 5. Real Mining Run

This cell writes the JSONL and Markdown audit files. It does not mutate compound data. Leave `RUN_REAL_LLM = False` for a safe dry demonstration.

In [6]:
AUDIT_JSONL = DATA_DIR / "raw_reactions" / "synonym_enrichment" / "notebook_candidates.jsonl"
AUDIT_MD = DATA_DIR / "raw_reactions" / "synonym_enrichment" / "notebook_candidates.md"

if RUN_REAL_LLM:
    summary = enricher.mine(
        PRESET_NAME,
        model=REAL_MODEL,
        rounds=DEFAULT_ROUNDS,
        min_votes=DEFAULT_MIN_VOTES,
        batch_size=DEFAULT_BATCH_SIZE,
        limit=20,
        audit_jsonl_fn=str(AUDIT_JSONL),
        audit_md_fn=str(AUDIT_MD),
        max_workers=20
    )
    pprint(summary)
else:
    print("Skipped real LLM mining. Set RUN_REAL_LLM = True to generate audit files.")
    print(f"Would write: {AUDIT_JSONL}")
    print(f"Would write: {AUDIT_MD}")

[18:19:50] INFO     Mining raw reaction synonym candidates: submitted 3 entries                        ]8;id=960953;file:///home/canary/Documents/Code/chemy/scripts/infra/logger.py\logger.py]8;;\:]8;id=233820;file:///home/canary/Documents/Code/chemy/scripts/infra/logger.py#45\45]8;;\

Output()

{'audit_entries': 20,
 'audit_jsonl': '/home/canary/Documents/Code/chemy/data/raw_reactions/synonym_enrichment/notebook_candidates.jsonl',
 'audit_markdown': '/home/canary/Documents/Code/chemy/data/raw_reactions/synonym_enrichment/notebook_candidates.md',
 'ready': 9,
 'unmapped_names': 20}


## 6. Review Audit Output

The Markdown file is for human review. The JSONL file is the apply source of truth. Entries with `status == "ready"` and `approved == true` are eligible for application, and apply re-checks guards before writing.

In [7]:
if AUDIT_JSONL.exists():
    audit_entries = chemy.store.load_jsonl(str(AUDIT_JSONL))
    print(f"Audit entries: {len(audit_entries)}")
    for entry in audit_entries[:5]:
        pprint({
            "name": entry.get("name"),
            "count": entry.get("count"),
            "status": entry.get("status"),
            "target_cid": entry.get("target_cid"),
            "target_name": entry.get("target_name"),
            "add_synonym": entry.get("add_synonym"),
            "candidate_synonyms": entry.get("candidate_synonyms", [])[:3],
        })
else:
    print("No notebook audit JSONL found yet.")

Audit entries: 20
{'add_synonym': None,
 'candidate_synonyms': [{'decision': 'no_db_match',
                         'norm_synonym': 'hydrogensulphide',
                         'raw_synonyms': [{'count': 5,
                                           'synonym': 'hydrogen sulphide'}],
                         'synonym': 'hydrogen sulphide',
                         'votes': 5}],
 'count': 319,
 'name': 'hydrogen sulfide',
 'status': 'no_unique_match',
 'target_cid': None,
 'target_name': None}
{'add_synonym': 'pyridinium chlorochromate',
 'candidate_synonyms': [{'decision': 'no_db_match',
                         'norm_synonym': 'pcc',
                         'raw_synonyms': [{'count': 5, 'synonym': 'PCC'}],
                         'synonym': 'PCC',
                         'votes': 5},
                        {'decision': 'matched',
                         'matched_cid': 6396768,
                         'matched_name': 'Dimethylsulfoxonium methylide',
                         'norm

## 7. Apply Approved Additions

Application mutates mapped compounds, so it is behind a second safety switch. The apply step reloads the audit JSONL and re-runs conflict/name guards before updating synonyms.

In [8]:
if RUN_APPLY:
    if not AUDIT_JSONL.exists():
        raise FileNotFoundError(AUDIT_JSONL)
    apply_summary = enricher.apply(str(AUDIT_JSONL))
    pprint(apply_summary)
else:
    print("Skipped apply. Set RUN_APPLY = True only after reviewing the audit JSONL/Markdown.")

Skipped apply. Set RUN_APPLY = True only after reviewing the audit JSONL/Markdown.


## 8. Equivalent CLI Commands

The notebook mirrors the CLI:

```bash
PYTHONPATH=. python3 -m scripts.run.raw_reaction_synonyms mine --preset wiki_crc_rp --limit 20
PYTHONPATH=. python3 -m scripts.run.raw_reaction_synonyms apply --audit-jsonl data/raw_reactions/synonym_enrichment/candidates.jsonl
```